In [1]:
# Base pyfantic

In [1]:
from __future__ import annotations

from copy import copy
from typing import (
    TYPE_CHECKING,
    Any,
    Dict,
    Optional,
    Tuple,
    Type,
    TypeVar,
    Union,
    get_args,
    get_origin,
)

from pydantic import BaseModel as _BaseModel, create_model
from pydantic_core import PydanticUndefined

try:
    from types import UnionType  # Python 3.10+: синтаксис int | None
except ImportError:
    UnionType = None

__all__ = (
    "BaseModel",
    "PartialBaseModel",
    "Partial",
    "create_partial",
)

In [2]:
if TYPE_CHECKING:
    Model = TypeVar("Model", bound="BaseModel")

In [3]:
_partial_models_cache: Dict[Tuple[Type[Any], str, Tuple[str, ...]], Type[Any]] = {}

In [4]:
class Partial:
    """Маркер для синтаксиса Model[Partial]."""

In [5]:
class BaseModel(_BaseModel):
    model_config = {"populate_by_name": True}

In [6]:
class PartialBaseModel(BaseModel):
    def __class_getitem__(cls: Type["Model"], item: Any) -> Type["Model"]:
        if item is Partial:
            return create_partial(cls)
        # Делегируем дальше, чтобы generics (Model[int] и т.п.) продолжали работать
        return super().__class_getitem__(item)

    @classmethod
    def partial(
        cls: Type["Model"], *field_names: str, name: Optional[str] = None
    ) -> Type["Model"]:
        return create_partial(cls, *field_names, name=name)

In [7]:
def _is_optional(annotation: Any) -> bool:
    origin = get_origin(annotation)
    if origin is Union or (UnionType is not None and origin is UnionType):
        return type(None) in get_args(annotation)
    return False

In [8]:
def create_partial(
    model: Type["Model"], *field_names: str, name: Optional[str] = None
) -> Type["Model"]:
    if name is None:
        name = f"{model.__name__}Partial"

    if not field_names:
        field_names = tuple(model.model_fields)

    cache_key = (model, name, tuple(sorted(field_names)))
    cached = _partial_models_cache.get(cache_key)
    if cached is not None:
        return cached

    overrides: Dict[str, Any] = {}
    for fn in field_names:
        info = model.model_fields[fn]
        annotation = info.annotation
        if not _is_optional(annotation):
            annotation = Optional[annotation]

        new_info = copy(info)  # сохраняем description, alias, examples и т.д.
        if new_info.default is PydanticUndefined and new_info.default_factory is None:
            new_info.default = None  # поле перестаёт быть обязательным

        overrides[fn] = (annotation, new_info)

    new_model = create_model(
        name,
        __base__=model,
        __module__=model.__module__,
        **overrides,
    )
    _partial_models_cache[cache_key] = new_model
    return new_model


In [ ]:
# Проверка работоспособности
# 
# 

In [9]:
from typing import Optional
from pydantic import Field

In [10]:
class User(PartialBaseModel):
    name: str = Field(description="Имя пользователя", alias="userName")
    age: int
    email: Optional[str] = None


In [12]:
UserPatch = User[Partial]
UserPatch

__main__.UserPartial

In [13]:
# Кэширование работает: один и тот же класс
assert UserPatch is User[Partial]
assert UserPatch is User.partial()

In [14]:
# Описание и алиас не потерялись
assert UserPatch.model_fields["name"].description == "Имя пользователя"

In [15]:
p = UserPatch(name="Иван")  # валидно — только одно поле
print(p.model_dump(exclude_unset=True))  # {'name': 'Иван'}

{'name': 'Иван'}
